In [ ]:
#@title 按這裡開始（先按 ▶）
# ←投影片未含，執行所需
print("✅ W17 出發！本週目標：把 Keras 模型轉成 TFLite、做 int8 量化、自己寫計時程式")
print("五段走完：訓練 → 轉檔 → 量化 → 量延遲 → 下判斷；前三段的產物都是「檔案」")
print("本週要自己補三個空：representative_dataset、invoke 那一行、decide() 的隱私判準")

# W17　邊緣 AI 與智慧物聯網應用（電腦教室版）

**開始之前**：按「複製到雲端硬碟」，改自己的副本才存得起來。

筆記本是**填空式**：看到 `____` 就是你要動手的地方。真的過不去再展開最後一格的「參考解」。

| 任務 | 你要自己寫的 | 產物 |
|---|---|---|
| 一 | （照跑） | `fmnist.keras`、`fmnist.tflite` |
| 二 | `representative_dataset` 那一行 | `fmnist_int8.tflite`＋三個檔案大小 |
| 三 | `bench()` 裡的推論那一行 | 兩個中位數毫秒 |
| 四 | （動腦）六個應用逐一判 | 邊緣雲端判斷表 |
| 五（進階） | `decide()` 的隱私判準 | 一個判斷函式 |

⚠️ **第 1 格沒跑成功，後面每一格都會報錯**，這一步一定要走完再往下。
本週不需要 GPU、不需要任何金鑰，模型訓練完之後就完全不吃網路。

### 第 1 格：訓練小模型並存檔

**這一格要做什麼**：沒有空格，整格照跑，只有 `epochs` 可以自己調大一點。
三輪大約一分鐘，正確率有八成以上就夠了。

**本週重點不在準確率，在「模型變成一個檔案」**：跑完請**點開左邊的檔案列表**，
親眼看到 `fmnist.keras` 出現——這是本週最重要的一個畫面。

**參數量怎麼算**：784×256 ＋ 256×128 ＋ 128×10 再加偏權值 ＝ **235146** 個參數。
每個參數 float32 佔 4 位元組，所以理論上 235146×4÷1024 ≈ **919 KB**，
等一下轉出來的 `.tflite` 會跟這個數字幾乎一樣。

**寫對了會看到什麼**：`verbose=2` 會印出三輪的正確率，跑完多一個 `.keras` 檔。

In [ ]:
#@title 第 1 格：訓練小模型並存檔
import tensorflow as tf
(x, y), (xt, yt) = tf.keras.datasets.fashion_mnist.load_data()
x, xt = x / 255.0, xt / 255.0
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="adam", metrics=["accuracy"],
              loss="sparse_categorical_crossentropy")
model.fit(x, y, epochs=3, batch_size=128, verbose=2)
model.save("fmnist.keras")

### 第 2 格：轉成 TFLite 並在本機跑一次

**這一格要做什麼**：沒有空格，整格照跑；轉檔完成會多一個 `.tflite` 檔案。

**TFLite 直譯器的四個步驟**（這四步在手機、樹莓派、微控制器上寫法完全一樣，
只是換一個直譯器實作，一定要記起來）：

1. `Interpreter(...)` — 建直譯器。
2. `allocate_tensors()` — 先把記憶體配好。
3. `set_tensor(...)` — 餵輸入。
4. `invoke()` → `get_tensor(...)` — 執行，取輸出。

**寫對了會看到什麼**：一個 0 到 9 的預測類別，左邊檔案列表多一個 `fmnist.tflite`。

**確認不吃任何服務**：推論全程沒有連任何 API——**這就是邊緣推論**，
對應理論那一段的飛航模式。

In [ ]:
#@title 第 2 格：轉成 TFLite 並在本機跑一次
conv = tf.lite.TFLiteConverter.from_keras_model(model)
tfl = conv.convert()
open("fmnist.tflite", "wb").write(tfl)
it = tf.lite.Interpreter(model_content=tfl)
it.allocate_tensors()
i0 = it.get_input_details()[0]
o0 = it.get_output_details()[0]
one = xt[:1].astype("float32")
it.set_tensor(i0["index"], one)
it.invoke()
print("預測類別 =", it.get_tensor(o0["index"])[0].argmax())

### 第 3 格：int8 量化與代表資料

**先分清楚兩件事**：`from_keras_model` **只是換格式**（大小幾乎不變）；
`optimizations` 才會真的把數字變小。兩件事不要混在一起。

- `optimizations = [Optimize.DEFAULT]`：打開量化。只給這一行叫**動態範圍量化**，
  量的是權重，啟動值推論時再轉回浮點。
- `representative_dataset`：轉檔器要**先看過真實資料**，
  才知道每一層數值的範圍。加上它才是完整整數量化。
- **不給代表資料的後果**：啟動值沒被量化，省下的空間與速度都打折扣。
- **代表資料不能用亂數**：範圍抓錯，量化後的準確率會掉得很誇張。

**這一格要做什麼**：補 `conv.representative_dataset` 那一行，指向上面那個函式。

⚠️ 最常見的錯是寫成 `rep_data()`：那會**先執行產生器再指派**，轉檔器要的是**函式本身**。
第二常見的是 `yield` 忘了包成 list——代表資料必須是一個 list，因為模型可能有多個輸入。

**寫對了會看到什麼**：三個檔案大小一起印出來，`int8` 那個大約剩四分之一。

**檔案大小紀錄表（自己填）**

| 檔案 | 你量到的大小 | 說明 |
|---|---|---|
| fmnist.keras | 　KB | Keras 壓縮檔，另含結構與訓練狀態 |
| fmnist.tflite | 　KB | 只換格式沒量化，每個參數 4 位元組 |
| fmnist_int8.tflite | 　KB | int8 量化，每個參數約 1 位元組 |
| 量化後剩幾分之一 | 　倍 | 第三列除以第二列，理論 1/4、實測約 1/3.8 |
| 省下多少 KB | 　KB | 第二列減第三列，寫成實際數字 |
| 參數量 | 235146 | 784×256＋256×128＋128×10 再加偏權值 |
| 理論 float32 大小 | 　KB | 參數量×4÷1024，和第二列比對 |

量化後**不會剛好剩四分之一**，因為結構描述與每一層的量化參數不會變小。

In [ ]:
#@title 第 3 格：int8 量化與代表資料
import os
def rep_data():
    for i in range(200):
        yield [xt[i:i+1].astype("float32")]

conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = ____   # ← 自己寫：上面那個函式
q8 = conv.convert()
open("fmnist_int8.tflite", "wb").write(q8)
for f in ["fmnist.keras", "fmnist.tflite", "fmnist_int8.tflite"]:
    print(f, round(os.path.getsize(f) / 1024, 1), "KB")

### 第 4 格：自己寫計時程式量推論延遲

**這一格要做什麼**：補一行——在 `set_tensor` 之後**執行一次推論**。

**計時的紀律（本週最重要的一段）**：

- 計時**只能包住推論那三行**，不能把建直譯器也包進去，否則量到的是**載入時間**。
- **回傳中位數，不回傳平均**：Colab 是共用機器，偶爾會有一兩次特別慢，
  平均會被一個離群值拉走。
- **先跑一次熱身再量**，是量測的基本紀律，做得完的人自己加一行。

**寫對了會看到什麼**：兩個毫秒數字（未量化、int8）。

⚠️ **要誠實看待第二個數字**：這個模型太小、電腦的 x86 浮點運算又太強，
`int8` 在電腦上常常**沒有變快，甚至略慢**。量化真正的好處要到手機、樹莓派或
微控制器上才看得出來。這反而是很好的教材：**量測要在目標裝置上做，
不能拿開發機的數字當結論。**

**延遲紀錄表（自己填）**

| 要記的項目 | 你量到的值 | 備註 |
|---|---|---|
| 未量化 中位數 | 　ms | 同一台機器、同一筆輸入才比得準 |
| int8 中位數 | 　ms | 在電腦的 x86 上不一定比較快 |
| 一秒能跑幾次 | 　次 | 1000 除以中位數 |
| 換成雲端要多久 | 　ms | 用理論例題的算法：上傳＋來回＋伺服器 |
| 雲端是本地的幾倍 | 　倍 | 雲端毫秒除以本地毫秒 |
| 第一次特別慢嗎 | 是／否 | 第一次要配記憶體，通常明顯較慢 |
| 和隔壁同學比 | 　ms | 同一份程式在不同機器上差多少 |

In [ ]:
#@title 第 4 格：自己寫 bench()
import time
def bench(path, n=100):
    it = tf.lite.Interpreter(model_path=path)
    it.allocate_tensors()
    i0 = it.get_input_details()[0]
    o0 = it.get_output_details()[0]
    x1 = xt[:1].astype("float32")
    ts = []
    for _ in range(n):
        s = time.perf_counter()
        it.set_tensor(i0["index"], x1)
        ____                # ← 自己寫：執行一次推論
        it.get_tensor(o0["index"])
        ts.append((time.perf_counter() - s) * 1000)
    ts.sort()
    return ts[n // 2]
print("未量化", round(bench("fmnist.tflite"), 3), "ms")
print("int8", round(bench("fmnist_int8.tflite"), 3), "ms")

### 任務四：六個應用，該放邊緣還是雲端（動腦，不用寫程式）

**判斷順序照流程圖走**（順序會影響答案，這是本週最好的討論題）：

1. 要不要即時？（延遲）
2. 含不含個資？（隱私）
3. 斷網要不要能用？（離線）
4. 模型塞不塞得下？（算力與記憶體）

口訣：**快、省、密、斷**。任何一項成立就先考慮放邊緣。

**六個應用逐一走一次**，並寫出它是**在哪一個菱形被攔下來的**：
手機臉部解鎖、路口車牌辨識、手錶跌倒偵測、AI 生圖工具、電商推薦排序、山區農地蟲害辨識。

**六題各一句理由，缺理由不計分。**

最後找出**兩邊都要**的那一個，寫出邊緣做什麼、雲端做什麼——
這是實務上最常見的做法：**邊緣負責「有沒有事」，雲端負責「到底是什麼事」。**

### 第 5 格（進階，任務五）：把判斷寫成函式

**這一格要做什麼**：補一行——資料敏感就放邊緣。

三個 `if` 的順序就是流程圖上三個菱形的順序，**換順序答案就會變**，
可以當堂討論為什麼延遲要排第一個。

**寫對了會看到什麼**：兩行判斷結果。
把任務四的六個應用**都加進 `apps` 跑一次**，看程式的答案和你們手寫的一不一樣。

**做完的人**：再把第四個判準（頻寬與費用）加進去。

In [ ]:
#@title 第 5 格（進階）：把判斷寫成函式
def decide(latency_ms, private, offline, model_mb):
    if latency_ms < 100:
        return "邊緣：要即時反應"
    if ____:                 # ← 自己寫：資料敏感就放邊緣
        return "邊緣：資料不出裝置"
    if offline:
        return "邊緣：沒網路也要能用"
    if model_mb > 100:
        return "雲端：模型太大塞不進去"
    return "雲端：算力大、好更新"

apps = [("手機臉部解鎖", 50, True, False, 5),
        ("AI 生圖工具", 5000, False, False, 4000)]
for a in apps:
    print(a[0], "→", decide(*a[1:]))

## 收工：延伸挑戰與繳交

- **A（每個人都要做完）換條件再量**：把 `bench` 的 `n` 從 100 改成 500 再量一次，
  看中位數有沒有變，並寫下你的解釋。
- **B 量準確率的代價**：用兩個 `.tflite` 各跑一千筆測試資料，
  比較量化前後正確率掉了幾個百分點。
- **C 說出為什麼**：說出為什麼量化能同時讓檔案變小又可能讓推論變快，
  並說出它的代價是什麼。

**電腦教室常見狀況**：
第 1 格下載 fashion_mnist 慢＝全班同時吃網路，等一下就好；
`NameError: model` ＝第 1 格沒跑完，回去重跑；
第 3 格轉檔卡住＝`representative_dataset` 寫成 `rep_data()` 了，去掉括號；
`ValueError: ... expects a list` ＝`yield` 忘了包成 list；
int8 沒有比較快＝這是正常的，寫進紀錄表當結論，不要改程式。

In [ ]:
#@title 收工檢查（直接按 ▶）
# ←投影片未含，執行所需
print("本週要交：三個檔案大小的紀錄表、自己寫的 bench() 與兩個中位數")
print("六個應用的邊緣雲端判斷與理由、寫完的 .ipynb（五格全部跑過）")
print("✅ 檔名：AI導論_W17_學號_姓名")
print("下週第 18 週期末考：紙筆測驗，範圍第 10 到 17 週，佔學期成績 20%")

---

<details>
<summary>參考解（三個空格都自己試過再打開）</summary>

```python
# 第 3 格
conv.representative_dataset = rep_data

# 第 4 格
it.invoke()

# 第 5 格
if private:
```

為什麼是這樣寫：

- **`rep_data` 不加括號**：轉檔器要的是**函式本身**，它會自己去呼叫。
  寫成 `rep_data()` 是先把產生器執行出來再指派，型別就不對了。
- **`yield` 一定要包成 list**：`yield [xt[i:i+1]...]`，
  因為模型可能有多個輸入，TFLite 一律用 list 接。
- **計時只包住 `set_tensor` → `invoke` → `get_tensor` 這三行**。
  把 `Interpreter(...)` 也包進去，量到的會是載入時間，數字會大一個量級。
- **中位數不是平均**：`ts.sort()` 之後取 `ts[n // 2]`。
  共用機器偶爾有一兩次特別慢，平均會被拉走，中位數不會。
- **`if private:` 排在第二個**：因為隱私是「資料能不能離開裝置」的硬條件，
  比離線與模型大小都優先。把它跟 `offline` 對調，
  某些應用的答案就會變——這正是當堂要討論的。

</details>